In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [2]:
# Uncomment this while running in Colab. Once it runs, enter your HuggingFace token ID.
# This is essential to access the gated Llama model.
# from huggingface_hub import notebook_login
# notebook_login()

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Use model ID if the pre-trained model isn't available locally.
model_id = 'meta-llama/Llama-3.2-1B'
pretrained_model_location = './tiny-stories/models/meta-llama-3_2-1B'
pretrained_model = AutoModelForCausalLM.from_pretrained(pretrained_model_location)

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 2212.20it/s, Materializing param=model.norm.weight]                              


In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [6]:
f'Vocab Size: {tokenizer.vocab_size}'

'Vocab Size: 128000'

In [7]:
pretrained_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [10]:
from datasets import load_from_disk

story_summary_pairs = load_from_disk('./tiny-stories/datasets/tiny-story-summaries-500k')

In [13]:
story_summary_pairs

Dataset({
    features: ['story', 'summary'],
    num_rows: 500000
})

In [14]:
story_summary_pairs['summary'][10]

'Tom, Mia, their mom and dad went to a pizza place where they ordered cheese and pepperoni pizzas. They enjoyed the food, admired the night sky, and later went home happily.'

In [15]:
def create_prompt_template(batched_dataset):
    return {
        'prompt': [f'{tokenizer.bos_token} {story} \nSummary: {summary} {tokenizer.eos_token}'
        for story, summary in zip(batched_dataset['story'], batched_dataset['summary'])]
    }

In [16]:
story_summary_pairs_with_prompt = story_summary_pairs.map(create_prompt_template, batched=True)

In [17]:
story_summary_pairs_with_prompt

Dataset({
    features: ['story', 'summary', 'prompt'],
    num_rows: 500000
})

In [18]:
story_summary_pairs_with_prompt['prompt'][10001]

'<|begin_of_text|> Once upon a time, there was a little bird. The bird was white and had a pretty song. One day, the bird saw a big pit. The pit was deep and dark. The bird wanted to see what was inside, so it flew down to take a look. As the bird got closer, the pit began to rise up and swallow the bird. The bird tried to fly away, but it was too late. The pit had taken the bird down. The moral of the story is that we should be careful when we see something dangerous. We should not go near it, even if we are curious. It is better to be safe than sorry. \nSummary: A curious bird flies down to look inside a deep pit and is swallowed by it, teaching a lesson to be careful with dangerous things. <|end_of_text|>'

In [19]:
def create_prompt_template(batched_dataset):
    return {
        'tokenized_prompt': [tokenizer.encode(prompt) for prompt in batched_dataset['prompt']]
    }

In [20]:
story_summary_pairs_with_prompt_tokenized =  story_summary_pairs_with_prompt.map(
    create_prompt_template, batched=True
)

In [21]:
story_summary_pairs_with_prompt_tokenized

Dataset({
    features: ['story', 'summary', 'prompt', 'tokenized_prompt'],
    num_rows: 500000
})

In [22]:
print(story_summary_pairs_with_prompt_tokenized['tokenized_prompt'][10])

[128000, 128000, 8529, 323, 61697, 4024, 311, 264, 23317, 2035, 449, 872, 3450, 323, 18233, 13, 2435, 1051, 30056, 323, 12304, 13, 2435, 7731, 520, 264, 2466, 2007, 323, 7111, 520, 279, 5130, 13, 578, 5130, 1047, 9364, 315, 2204, 88870, 13, 8529, 15262, 17604, 23317, 323, 61697, 15262, 25349, 21446, 23317, 13, 330, 3923, 656, 499, 1390, 311, 8343, 7673, 3450, 4691, 13, 330, 40, 1390, 17604, 23317, 9135, 8529, 1071, 13, 330, 40, 1390, 25349, 21446, 23317, 9135, 61697, 1071, 13, 330, 33413, 11, 584, 690, 2015, 832, 17604, 23317, 323, 832, 25349, 21446, 23317, 1359, 18233, 1071, 13, 1283, 6688, 279, 5130, 311, 279, 68269, 13, 32862, 11, 279, 88870, 3782, 13, 2435, 86798, 1695, 323, 7111, 76454, 13, 22969, 323, 18233, 1511, 264, 2466, 22145, 311, 4018, 279, 88870, 1139, 35354, 13, 2435, 6688, 8529, 323, 61697, 1855, 264, 16363, 389, 264, 12235, 13, 330, 3513, 16994, 11, 279, 23317, 374, 4106, 1359, 3450, 1071, 13, 8529, 323, 61697, 42423, 389, 872, 23317, 311, 7155, 433, 1523, 13, 5112, 81

In [23]:
tokenizer.encode('...')

[128000, 1131]

In [24]:
tokenizer.pad_token_id = 1131
tokenizer.pad_token

'...'

In [25]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TinyStoryDataset(Dataset):
    def __init__(self, tokenized_list):
        self.data = tokenized_list['tokenized_prompt']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.long, device=device)

In [26]:
tokenized_dataset = TinyStoryDataset(story_summary_pairs_with_prompt_tokenized)

In [27]:
from torch.nn.utils.rnn import pad_sequence

def dynamic_collate_fn(batch):
    pad_token_id = tokenizer.pad_token_id

    # 1. Pad the sequences so they all match the longest one in this batch
    padded_input_ids = pad_sequence(batch, batch_first=True, padding_value=pad_token_id)

    # 2. For Causal LM, labels are a direct copy of the input_ids
    labels = padded_input_ids.clone()

    # 3. Dynamically create the attention mask
    # It creates a tensor of 1s where the token is real, and 0s where it is padding
    attention_masks = (padded_input_ids != pad_token_id).long()

    # 4. Replace padding token ids in labels with -100 to ignore them in loss calculation
    labels[labels == pad_token_id] = -100

    # 3. Return the neat dictionary that standard NLP models expect
    return {
        'input_ids': padded_input_ids,
        'attention_mask': attention_masks,
        'labels': labels
    }

In [28]:
train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=dynamic_collate_fn
)

In [29]:
first_batch = next(iter(train_dataloader))
first_batch['input_ids'].shape, first_batch['attention_mask'].shape, first_batch['labels'].shape

(torch.Size([2, 254]), torch.Size([2, 254]), torch.Size([2, 254]))

In [30]:
print(tokenizer.decode(first_batch['input_ids'].tolist()[0]))
print('\n')
print(tokenizer.decode([label if label != -100 else tokenizer.pad_token_id for label in first_batch['labels'].tolist()[0]]))

<|begin_of_text|><|begin_of_text|> Once upon a time, a boy named Timmy was playing with his toys. He had a red car, a blue ball, and a yellow truck. Timmy loved to play with his toys, but he had a problem. His little sister, Lily, always wanted to play with his toys too.  One day, Timmy decided to hide his favorite toy, a green dinosaur, so that Lily couldn't play with it. He put it in his closet and closed the door. But when he came back later, he saw that Lily had found the dinosaur and was playing with it. Timmy was angry and sad.  He wanted to get his dinosaur back, but he didn't want to be mean to Lily. So he decided to lead her to another toy, a sour candy. He knew she would love it because she liked sour things. He gave her the candy and while she was eating it, he took the dinosaur back. Timmy felt happy again, and he learned that sharing is better than hiding objects. Summary: Timmy hides his favorite toy from his sister but realizes that sharing is better than hiding objects.

In [31]:
pretrained_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [32]:
import math

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank, dtype=torch.bfloat16))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim, dtype=torch.bfloat16))
        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

In [33]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [34]:
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_with_lora(module, rank, alpha)

In [35]:
# Freeze the model parameters.
total_params = sum(p.numel() for p in pretrained_model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')

for param in pretrained_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in pretrained_model.parameters() if p.requires_grad)
print(f'Total trainable parameters after freezing the model: {total_params:,}')

Total trainable parameters: 1,235,814,400
Total trainable parameters after freezing the model: 0


In [36]:
replace_linear_with_lora(pretrained_model, rank=16, alpha=16)

total_params = sum(p.numel() for p in pretrained_model.parameters() if p.requires_grad)
print(f'Total trainable parameters after applying LoRA: {total_params:,}')

Total trainable parameters after applying LoRA: 13,357,056


In [37]:
pretrained_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=2048, bias=False)
            (lora): LoRALayer()
          )
          (k_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=512, bias=False)
            (lora): LoRALayer()
          )
          (v_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=512, bias=False)
            (lora): LoRALayer()
          )
          (o_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=2048, bias=False)
            (lora): LoRALayer()
          )
        )
        (mlp): LlamaMLP(
          (gate_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=8192, bias=False)
            (lora): LoRALayer()


In [ ]:
pretrained_model.to(device)

In [39]:
import torch

def generate_content(prompt):
    encoded_ids = tokenizer.encode(prompt)
    encoded_ids = torch.tensor(encoded_ids, dtype=torch.int64, device=device)

    encoded_ids = encoded_ids.view(1, encoded_ids.shape[-1])

    generated_ids = pretrained_model.generate(
        encoded_ids, max_length=500, pad_token_id=tokenizer.eos_token_id
    )

    print(tokenizer.decode(generated_ids)[0])

In [40]:
# Checking the initial model performance in un-affected
generate_content('Hey, how are you')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|>Hey, how are you? Today I am going to show you how to make a 3d model of a beautiful flower. I have used a lot of different 3d modeling programs and I think that this is the best way to make a 3d model of a flower. I will explain you how to make a 3d model of a flower in 3 easy steps.
1. First of all you need to create a flower. I have used a flower from my collection. I have made a 3d model of it and then I have created a 3d model of the flower.
2. Now you need to create a flower in 3d. I have used a 3d modeling program. I have created a flower in 3d and then I have created a 3d model of the flower.
3. Now you need to create a 3d model of a flower. I have used a 3d modeling program. I have created a flower in 3d and then I have created a 3d model of the flower.
4. Now you need to create a 3d model of a flower. I have used a 3d modeling program. I have created a flower in 3d and then I have created a 3d model of the flower.
5. Now you need to create a 3d model of a flo

In [41]:
from torch.optim import AdamW
from transformers import get_scheduler

# Standard learning rate for fine-tuning
learning_rate = 5e-5
num_epochs = 1

optimizer = AdamW(pretrained_model.parameters(), lr=learning_rate)

# Calculate total training steps
num_training_steps = num_epochs * story_summary_pairs_with_prompt_tokenized.num_rows

# Set up the learning rate scheduler
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps), desc="Training")
pretrained_model.train() # Put model in training mode

losses = []

for epoch in range(num_epochs):
    for batch in train_dataloader:
        # if batch['input_ids'].shape[1] > 511:
        #     continue

        # Move all tensors in the batch to the designated device
        batch = {k: v for k, v in batch.items()}

        # 1. Forward pass
        # print('Forward Pass...')
        outputs = pretrained_model(**batch)
        loss = outputs.loss

        # 2. Backward pass
        # print(f'Back propagate...: {loss}')
        loss.backward()
        losses.append(loss)

        # 3. Update weights and learning rate
        # print('Taking a step back')
        optimizer.step()
        lr_scheduler.step()

        # 4. Clear gradients for the next step
        # print('Clear gradients...')
        optimizer.zero_grad()

        # Update progress bar and print loss
        # print('Updating progress bar...')
        progress_bar.update(1)
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        # break

In [44]:
story_summary_pairs_with_prompt_tokenized['story'][1001]

"Once upon a time, there was a little girl named Lily. She loved to play outside in the park. One day, Lily saw a big swing and she wanted to try it. But she was too scared to swing high. Lily's mom said they had to go home soon, so Lily ran up the stairs to go down the slide one last time. As she was going down the slide, she saw a lady with a basket of apples. The lady told Lily that the apples were delicious and gave her one to try. Lily took a bite of the apple and it was so yummy! She felt happy and brave. So, she ran back to the swing and started to swing higher and higher. She felt like she could touch the sky! From that day on, Lily loved to swing and always remembered the delicious apple that gave her courage. Summary: Lily overcomes her fear of swinging high after trying a delicious apple given to her by a kind lady in the park."

In [45]:
f'{tokenizer.bos_token} {story_summary_pairs_with_prompt_tokenized['story'][10001]} \nSummary: '

'<|begin_of_text|> Once upon a time, there was a little bird. The bird was white and had a pretty song. One day, the bird saw a big pit. The pit was deep and dark. The bird wanted to see what was inside, so it flew down to take a look. As the bird got closer, the pit began to rise up and swallow the bird. The bird tried to fly away, but it was too late. The pit had taken the bird down. The moral of the story is that we should be careful when we see something dangerous. We should not go near it, even if we are curious. It is better to be safe than sorry. \nSummary: '

In [56]:
generate_content(f'{tokenizer.bos_token} {story_summary_pairs_with_prompt_tokenized['story'][50001]} \nSummary: ')

<|begin_of_text|><|begin_of_text|> Once upon a time, there was a little girl named Lily. She loved dressing up in her favorite costumes and pretending to be different characters. One day, Lily's mom bought her an original superhero costume. Lily was so excited and couldn't wait to show her friends. Lily went to her friend's house to play. She showed her friends her new superhero costume and they all loved it. But then, one of her friends said that her costume was better. Lily didn't like that and it made her feel sad.  Lily decided to show her friends how fast she could run in her new costume. She ran so fast that she didn't see the big rock in front of her. She tripped and fell, and her costume ripped. Lily started crying and her friends tried to help her, but it was too late.  Lily's mom had to buy her a new costume, but it wasn't the same as the original one. Lily learned that it's not important to have the best costume or be the fastest, but to have fun and be safe. 
Summary:  Lily